In [1]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from keras.utils import to_categorical
import numpy as np
from sklearn.svm import SVC
import pandas as pd
#from sklearn.datasets import load_iris


In [2]:
def dataset_smote(beta, ph):
    #import
    X_df = pd.read_csv(beta, delimiter=' ')
    y_df = pd.read_csv(ph, delimiter=' ')
    X = X_df.values
    y = y_df['label'].values

    #smote
    sm = SMOTE(random_state=42) #set random seet for reproducibility
    X_res, y_res = sm.fit_resample(X, y)
    X_res = np.round(X_res, decimals = 5)
    #y_res = to_categorical(y_res)
    # Print the class distribution before and after SMOTE
    print("Original class distribution:")
    print({i: list(y).count(i) for i in range(len(set(y)))})
    print("Class distribution after SMOTE:")
    print({i: list(y_res).count(i) for i in range(len(set(y_res)))})
    return X_df, X_res, y_res

def dataset_smote_onehot(beta, ph):
    #import
    X_df = pd.read_csv(beta, delimiter=' ')
    y_df = pd.read_csv(ph, delimiter=' ')
    X = X_df.values
    y = y_df['label'].values

    #smote
    sm = SMOTE(random_state=42) #set random seet for reproducibility
    X_res, y_res = sm.fit_resample(X, y)
    X_res = np.round(X_res, decimals = 5)
    y_res = to_categorical(y_res)
    # Print the class distribution before and after SMOTE
    # Print the class distribution before and after SMOTE
    print("Original class distribution:")
    print({i: np.sum(y == i) for i in np.unique(y)})
    print("Class distribution after SMOTE:")
    print({i: np.sum(y_res[:, i]) for i in range(y_res.shape[1])})
    return X_res, y_res

def default_svm(kernel='linear', C=1):
    """
    Create an SVM model with specified kernel and regularization parameter.

    Parameters:
    kernel: Kernel type (default is 'linear').
    C: Regularization parameter (default is 1).

    Returns:
    svm: SVM model.
    """
    svm = SVC(kernel=kernel, C=C, random_state=42)
    return svm

def skf_process(model, X, y, n_splits=10):
    """
    Perform stratified k-fold cross-validation.

    Parameters:
    model: The machine learning model to train.
    X: Feature matrix.
    y: Target vector.
    n_splits: Number of folds for cross-validation (default is 10).

    Returns:
    A dictionary with lists of metrics for each fold.
    """
    # Step 4: Set up StratifiedKFold for cross-validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Initialize lists to store metrics for each fold
    accuracy_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []

    # Step 5: Train and evaluate the model for each fold
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Train the model
        model.fit(X_train, y_train)

        # Predict the labels for the test set
        y_pred = model.predict(X_test)

        # Calculate evaluation metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        # Append the metrics to the respective lists
        accuracy_scores.append(accuracy)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)

        # Print metrics for the current fold
        print(f"Fold {len(accuracy_scores)}:")
        print(f"Accuracy: {accuracy:.3f}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1 Score: {f1:.3f}")
        print("-" * 30)

    return {
        "accuracy": accuracy_scores,
        "precision": precision_scores,
        "recall": recall_scores,
        "f1": f1_scores
    }

def avg_std(metrics):
    """
    Calculate and print the average and standard deviation of metrics.

    Parameters:
    metrics: A dictionary containing lists of metrics for each fold.
    """
    avg_accuracy = np.mean(metrics['accuracy'])
    std_accuracy = np.std(metrics['accuracy'])
    avg_precision = np.mean(metrics['precision'])
    std_precision = np.std(metrics['precision'])
    avg_recall = np.mean(metrics['recall'])
    std_recall = np.std(metrics['recall'])
    avg_f1 = np.mean(metrics['f1'])
    std_f1 = np.std(metrics['f1'])

    print("\nAverage Scores Across all Folds:")
    print(f"Accuracy: {avg_accuracy:.3f} +- {std_accuracy:.3f}")
    print(f"Precision: {avg_precision:.3f} +- {std_precision:.3f}")
    print(f"Recall: {avg_recall:.3f} +- {std_recall:.3f}")
    print(f"F1-score: {avg_f1:.3f} +- {std_f1:.3f}")



In [19]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/TA_KIM/data')
dir = os.getcwd()
beta = f'{dir}/beta_p56_01_noppi.txt'
ph = f'{dir}/ph_label_universal.txt'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# load dataset
df, X, y = dataset_smote(beta, ph)

Original class distribution:
{0: 100, 1: 46, 2: 167, 3: 120}
Class distribution after SMOTE:
{0: 167, 1: 167, 2: 167, 3: 167}


In [21]:
# Initialize the SVM classifier
svm_linear = default_svm(kernel='linear', C=1)

In [6]:
# Perform stratified 10-fold cross-validation
metrics_linear = skf_process(svm_linear, X, y, n_splits=10)


Fold 1:
Accuracy: 0.821
Precision: 0.832
Recall: 0.821
F1 Score: 0.812
------------------------------
Fold 2:
Accuracy: 0.791
Precision: 0.789
Recall: 0.791
F1 Score: 0.790
------------------------------
Fold 3:
Accuracy: 0.851
Precision: 0.847
Recall: 0.851
F1 Score: 0.848
------------------------------
Fold 4:
Accuracy: 0.821
Precision: 0.815
Recall: 0.821
F1 Score: 0.810
------------------------------
Fold 5:
Accuracy: 0.851
Precision: 0.848
Recall: 0.851
F1 Score: 0.847
------------------------------
Fold 6:
Accuracy: 0.836
Precision: 0.834
Recall: 0.836
F1 Score: 0.828
------------------------------
Fold 7:
Accuracy: 0.821
Precision: 0.811
Recall: 0.821
F1 Score: 0.812
------------------------------
Fold 8:
Accuracy: 0.806
Precision: 0.799
Recall: 0.806
F1 Score: 0.801
------------------------------
Fold 9:
Accuracy: 0.803
Precision: 0.802
Recall: 0.803
F1 Score: 0.790
------------------------------
Fold 10:
Accuracy: 0.848
Precision: 0.849
Recall: 0.848
F1 Score: 0.846
----------

In [7]:
# Print average and standard deviation of the metrics
avg_std(metrics_linear)


Average Scores Across all Folds:
Accuracy: 0.825 +- 0.020
Precision: 0.822 +- 0.021
Recall: 0.825 +- 0.020
F1-score: 0.818 +- 0.021


<br>

**One hot encoding**

In [8]:
# load dataset
X_hot, y_hot = dataset_smote_onehot(beta, ph)
# Initialize the SVM classifier
#svm_linear = default_svm(kernel='linear', C=1)
# Perform stratified 10-fold cross-validation
metrics_linear_onehot = skf_process(svm_linear, X, y, n_splits=10)

Original class distribution:
{0: 100, 1: 46, 2: 167, 3: 120}
Class distribution after SMOTE:
{0: 167.0, 1: 167.0, 2: 167.0, 3: 167.0}
Fold 1:
Accuracy: 0.821
Precision: 0.832
Recall: 0.821
F1 Score: 0.812
------------------------------
Fold 2:
Accuracy: 0.791
Precision: 0.789
Recall: 0.791
F1 Score: 0.790
------------------------------
Fold 3:
Accuracy: 0.851
Precision: 0.847
Recall: 0.851
F1 Score: 0.848
------------------------------
Fold 4:
Accuracy: 0.821
Precision: 0.815
Recall: 0.821
F1 Score: 0.810
------------------------------
Fold 5:
Accuracy: 0.851
Precision: 0.848
Recall: 0.851
F1 Score: 0.847
------------------------------
Fold 6:
Accuracy: 0.836
Precision: 0.834
Recall: 0.836
F1 Score: 0.828
------------------------------
Fold 7:
Accuracy: 0.821
Precision: 0.811
Recall: 0.821
F1 Score: 0.812
------------------------------
Fold 8:
Accuracy: 0.806
Precision: 0.799
Recall: 0.806
F1 Score: 0.801
------------------------------
Fold 9:
Accuracy: 0.803
Precision: 0.802
Recall: 0

In [9]:
# Print average and standard deviation of the metrics
avg_std(metrics_linear_onehot)


Average Scores Across all Folds:
Accuracy: 0.825 +- 0.020
Precision: 0.822 +- 0.021
Recall: 0.825 +- 0.020
F1-score: 0.818 +- 0.021


In [10]:
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectFromModel
import numpy as np

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Initialize lists to store feature importance scores
feature_scores = []

# Iterate over each fold
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Initialize SVM with linear kernel
    svm = SVC(kernel='linear', C=1.0, random_state=42)

    # Train the SVM model
    svm.fit(X_train, y_train)

    # Extract feature importance scores
    feature_scores.append(np.abs(svm.coef_))

# Calculate average feature importance scores across all folds
avg_feature_scores = np.mean(feature_scores, axis=0)

# Select features based on importance scores
selector = SelectFromModel(estimator=svm, prefit=True)
X_selected = selector.transform(X)

# Print the average feature importance scores
print("Average Feature Importance Scores:")
print(avg_feature_scores)

# Print the selected features
print("Selected Features:")
print(X_selected)


Average Feature Importance Scores:
[[0.05309132 0.51694014 0.97944938 0.46047721]
 [0.02331821 0.1785179  0.53263622 0.28336946]
 [0.61148662 0.82706012 1.99560063 2.05534311]]
Selected Features:
[[1.4 0.2]
 [1.4 0.2]
 [1.3 0.2]
 [1.5 0.2]
 [1.4 0.2]
 [1.7 0.4]
 [1.4 0.3]
 [1.5 0.2]
 [1.4 0.2]
 [1.5 0.1]
 [1.5 0.2]
 [1.6 0.2]
 [1.4 0.1]
 [1.1 0.1]
 [1.2 0.2]
 [1.5 0.4]
 [1.3 0.4]
 [1.4 0.3]
 [1.7 0.3]
 [1.5 0.3]
 [1.7 0.2]
 [1.5 0.4]
 [1.  0.2]
 [1.7 0.5]
 [1.9 0.2]
 [1.6 0.2]
 [1.6 0.4]
 [1.5 0.2]
 [1.4 0.2]
 [1.6 0.2]
 [1.6 0.2]
 [1.5 0.4]
 [1.5 0.1]
 [1.4 0.2]
 [1.5 0.2]
 [1.2 0.2]
 [1.3 0.2]
 [1.4 0.1]
 [1.3 0.2]
 [1.5 0.2]
 [1.3 0.3]
 [1.3 0.3]
 [1.3 0.2]
 [1.6 0.6]
 [1.9 0.4]
 [1.4 0.3]
 [1.6 0.2]
 [1.4 0.2]
 [1.5 0.2]
 [1.4 0.2]
 [4.7 1.4]
 [4.5 1.5]
 [4.9 1.5]
 [4.  1.3]
 [4.6 1.5]
 [4.5 1.3]
 [4.7 1.6]
 [3.3 1. ]
 [4.6 1.3]
 [3.9 1.4]
 [3.5 1. ]
 [4.2 1.5]
 [4.  1. ]
 [4.7 1.4]
 [3.6 1.3]
 [4.4 1.4]
 [4.5 1.5]
 [4.1 1. ]
 [4.5 1.5]
 [3.9 1.1]
 [4.8 1.8]
 [4.  1.3]
 [4.9 1.5]
 

In [11]:
def skf_process_relevant(model, X, y, n_splits=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    feature_scores = np.zeros((n_splits, X.shape[1]))

    accuracy_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []

    for fold, (train_index, test_index) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        accuracy_scores.append(accuracy)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)

        if hasattr(model, 'coef_'):
            feature_scores[fold - 1] = np.abs(model.coef_).mean(axis=0)

        print(f"Fold {fold}:")
        print(f"Accuracy: {accuracy:.3f}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1 Score: {f1:.3f}")
        print("-" * 30)

    avg_metrics = {
        "accuracy": np.mean(accuracy_scores),
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "f1": np.mean(f1_scores)
    }

    avg_feature_scores = np.mean(feature_scores, axis=0)
    top_100_indices = np.argsort(avg_feature_scores)[-100:]
    top_100_scores = avg_feature_scores[top_100_indices]

    return avg_metrics, top_100_indices, top_100_scores

def indices_to_genes_with_scores(df, top_indices, top_scores):
    top_genes = df.columns[top_indices]
    top_genes_with_scores = pd.DataFrame({"Gene": top_genes, "Score": top_scores})
    return top_genes_with_scores


In [12]:
# Initialize your SVM model
model = SVC(kernel='linear', C=1, random_state=42)

# Perform stratified k-fold cross-validation and get top 100 relevant features
avg_metrics, top_100_indices, top_100_scores = skf_process_relevant(model, X, y)



Fold 1:
Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
------------------------------
Fold 2:
Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
------------------------------
Fold 3:
Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
------------------------------
Fold 4:
Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
------------------------------
Fold 5:
Accuracy: 0.933
Precision: 0.944
Recall: 0.933
F1 Score: 0.933
------------------------------
Fold 6:
Accuracy: 0.867
Precision: 0.905
Recall: 0.867
F1 Score: 0.861
------------------------------
Fold 7:
Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
------------------------------
Fold 8:
Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
------------------------------
Fold 9:
Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
------------------------------
Fold 10:
Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
----------

In [13]:
# Display average metrics across all folds
print("\nAverage Scores Across all Folds:")
print(f"Accuracy: {avg_metrics['accuracy']:.3f}")
print(f"Precision: {avg_metrics['precision']:.3f}")
print(f"Recall: {avg_metrics['recall']:.3f}")
print(f"F1-score: {avg_metrics['f1']:.3f}")


Average Scores Across all Folds:
Accuracy: 0.980
Precision: 0.985
Recall: 0.980
F1-score: 0.979


In [14]:
# Convert indices to gene names with scores
top_genes_with_scores = indices_to_genes_with_scores(df, top_100_indices, top_100_scores)

# Display the DataFrame with top genes and their scores
print(top_genes_with_scores)

      Gene     Score
0     AAK1  0.229299
1    ABCA4  0.507506
2   ABLIM1  0.933063
3  ABHD12B  1.169229


In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import LinearSVC

def skf_process_relevant(model, X, y, n_splits=10):
    """
    Perform stratified k-fold cross-validation and extract top 100 relevant features.

    Parameters:
    model: The machine learning model to train.
    X: Feature matrix.
    y: Target vector.
    n_splits: Number of folds for cross-validation (default is 10).

    Returns:
    avg_metrics: A dictionary with average metrics across all folds.
    top_100_indices: Indices of the top 100 relevant features.
    top_100_scores: Scores of the top 100 relevant features.
    """
    # Initialize StratifiedKFold
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Initialize arrays to store feature importance scores
    feature_scores = np.zeros((n_splits, X.shape[1]))

    # Initialize lists to store metrics for each fold
    accuracy_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []

    # Train and evaluate the model for each fold
    for fold, (train_index, test_index) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Train the model
        model.fit(X_train, y_train)

        # Predict the labels for the test set
        y_pred = model.predict(X_test)

        # Calculate evaluation metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        # Append the metrics to the respective lists
        accuracy_scores.append(accuracy)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)

        # Extract feature importance scores
        if hasattr(model, 'coef_'):
            feature_scores[fold - 1] = np.abs(model.coef_).mean(axis=0)

        # Print metrics for the current fold
        print(f"Fold {fold}:")
        print(f"Accuracy: {accuracy:.3f}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1 Score: {f1:.3f}")
        print("-" * 30)

    # Calculate average metrics across all folds
    avg_metrics = {
        "accuracy": np.mean(accuracy_scores),
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "f1": np.mean(f1_scores)
    }

    # Calculate average feature importance scores across all folds
    avg_feature_scores = np.mean(feature_scores, axis=0)

    # Select the top 100 features based on their importance scores
    top_100_indices = np.argsort(avg_feature_scores)[-100:][::-1]
    top_100_scores = avg_feature_scores[top_100_indices]

    return avg_metrics, top_100_indices, top_100_scores


def indices_to_genes_with_scores(df, top_indices, top_scores):
    """
    Convert indices of the top relevant features to gene names with scores.

    Parameters:
    df: pandas.DataFrame - DataFrame containing feature names (genes).
    top_indices: numpy.ndarray - Indices of the top relevant features.
    top_scores: numpy.ndarray - Scores of the top relevant features.

    Returns:
    top_genes_with_scores: pandas.DataFrame - DataFrame with gene names and scores.
    """
    top_genes = df.columns[top_indices]
    top_genes_with_scores = pd.DataFrame({"Gene": top_genes, "Score": top_scores})
    top_genes_with_scores = top_genes_with_scores.sort_values(by="Score", ascending=False).reset_index(drop=True)
    return top_genes_with_scores



In [22]:

# Perform stratified k-fold cross-validation and get top 100 relevant features
avg_metrics, top_100_indices, top_100_scores = skf_process_relevant(model, X, y)

# Display average metrics across all folds
print("\nAverage Scores Across all Folds:")
print(f"Accuracy: {avg_metrics['accuracy']:.3f}")
print(f"Precision: {avg_metrics['precision']:.3f}")
print(f"Recall: {avg_metrics['recall']:.3f}")
print(f"F1-score: {avg_metrics['f1']:.3f}")


Fold 1:
Accuracy: 0.821
Precision: 0.832
Recall: 0.821
F1 Score: 0.812
------------------------------
Fold 2:
Accuracy: 0.791
Precision: 0.789
Recall: 0.791
F1 Score: 0.790
------------------------------
Fold 3:
Accuracy: 0.851
Precision: 0.847
Recall: 0.851
F1 Score: 0.848
------------------------------
Fold 4:
Accuracy: 0.821
Precision: 0.815
Recall: 0.821
F1 Score: 0.810
------------------------------
Fold 5:
Accuracy: 0.851
Precision: 0.848
Recall: 0.851
F1 Score: 0.847
------------------------------
Fold 6:
Accuracy: 0.836
Precision: 0.834
Recall: 0.836
F1 Score: 0.828
------------------------------
Fold 7:
Accuracy: 0.821
Precision: 0.811
Recall: 0.821
F1 Score: 0.812
------------------------------
Fold 8:
Accuracy: 0.806
Precision: 0.799
Recall: 0.806
F1 Score: 0.801
------------------------------
Fold 9:
Accuracy: 0.803
Precision: 0.802
Recall: 0.803
F1 Score: 0.790
------------------------------
Fold 10:
Accuracy: 0.848
Precision: 0.849
Recall: 0.848
F1 Score: 0.846
----------

In [23]:
# Assuming `df` is your DataFrame containing gene names as columns
top_genes_with_scores = indices_to_genes_with_scores(df, top_100_indices, top_100_scores)

# Adjust pandas display options to show the full DataFrame
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Display the DataFrame with top genes and their scores
print(top_genes_with_scores)

        Gene     Score
0      PAPLN  0.377753
1       PLD1  0.377518
2     S100A2  0.348442
3      SMPD3  0.325525
4      GSTM5  0.309012
5     RAB22A  0.302306
6       PIGB  0.301603
7      CDKL2  0.298688
8    COLEC11  0.291266
9     GPRIN2  0.278451
10     KCNQ1  0.277392
11       NDN  0.270174
12     HOXD4  0.268228
13     ITM2C  0.265541
14      ENO1  0.265461
15  C17orf69  0.258681
16     PRR15  0.252129
17     SPAG6  0.248067
18      VAX2  0.247823
19     OVOL1  0.246386
20      TC2N  0.245489
21     DLEU7  0.243323
22      OSR2  0.242039
23    ATPGD1  0.238359
24      DMP1  0.238071
25       GGN  0.237091
26     LOXL3  0.236665
27     THOC6  0.235552
28     PITX3  0.234488
29  C17orf99  0.232536
30    NEURL3  0.229055
31     HOXD3  0.228649
32    HMGXB4  0.226499
33  C17orf85  0.225752
34      SGCE  0.225673
35   GAL3ST3  0.224782
36     ANXA2  0.223049
37      CES4  0.221666
38  C1orf100  0.221029
39  C10orf28  0.220725
40    HOXA10  0.220638
41    MIR10B  0.219348
42     NCOA

In [26]:
top_genes_with_scores.to_csv('top_genes_svm.txt', sep=' ', index=False)